# Week 2 — Data Preprocessing, Baseline Model, Plots
**Dataset:** Daily and Sports Activities Dataset (UCI)

**Tasks this week:**
1. Download and load the dataset
2. Normalize sensor data
3. Split by subject (no data leakage)
4. Train Random Forest baseline
5. Evaluate and plot results

## 0. Install dependencies

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn requests -q

## 1. Download and Load the Dataset

In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd

# ── Download ──────────────────────────────────────────────────────────────────
DATA_URL = "https://archive.ics.uci.edu/static/public/256/daily+and+sports+activities.zip"
ZIP_PATH = "daily_sports.zip"
EXTRACT_DIR = "daily_sports_data"

if not os.path.exists(ZIP_PATH):
    print("Downloading dataset ...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Done.")
else:
    print("Zip already present.")

if not os.path.exists(EXTRACT_DIR):
    print("Extracting ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Done.")
else:
    print("Already extracted.")

In [ ]:
# ── Find the root of the data ────────────────────────────────────────────────
# The archive contains a folder 'data/' with sub-folders a01..a19 (activities)
# Inside each: p1..p8 (persons), each person has s01..s60 (segments), each
# segment is a 125-row × 45-column CSV (5 body units × 9 channels × 125 samples).

import glob

# Locate all segment files
pattern = os.path.join(EXTRACT_DIR, "**", "*.txt")
all_files = sorted(glob.glob(pattern, recursive=True))
print(f"Total segment files found: {len(all_files)}")
print("Example path:", all_files[0] if all_files else "NONE")

In [ ]:
# ── Parse file paths and load data ───────────────────────────────────────────
# Expected path pattern: .../data/a<ACT>/p<PERSON>/s<SEG>.txt

records = []
for fpath in all_files:
    parts = fpath.replace("\\", "/").split("/")
    # Find the parts that match a<nn>, p<n>, s<nn>.txt
    try:
        act_part  = [p for p in parts if p.startswith("a") and p[1:].isdigit()][0]
        per_part  = [p for p in parts if p.startswith("p") and p[1:].isdigit()][0]
        activity  = int(act_part[1:])   # 1-19
        person    = int(per_part[1:])   # 1-8
    except IndexError:
        continue  # skip unexpected paths

    data = np.loadtxt(fpath, delimiter=",")  # shape (125, 45)
    records.append((activity, person, data))

print(f"Loaded {len(records)} segments.")
print(f"Segment shape: {records[0][2].shape}")

In [ ]:
# ── Build flat feature matrix (flatten each 125×45 segment → 5625 features) ──
# For the Random Forest baseline we use statistical features per channel:
# mean, std, min, max, range  →  5 stats × 45 channels = 225 features

X_list, y_list, subject_list = [], [], []

for activity, person, seg in records:
    feats = np.concatenate([
        seg.mean(axis=0),
        seg.std(axis=0),
        seg.min(axis=0),
        seg.max(axis=0),
        seg.max(axis=0) - seg.min(axis=0),
    ])  # length = 225
    X_list.append(feats)
    y_list.append(activity - 1)   # 0-indexed labels
    subject_list.append(person)

X = np.array(X_list)          # (N, 225)
y = np.array(y_list)          # (N,)
subjects = np.array(subject_list)  # (N,)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Classes : {np.unique(y)}")
print(f"Subjects: {np.unique(subjects)}")

## 2. Data Preprocessing — Normalisation & Subject-aware Split

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

# ── Subject-aware split (no data leakage) ────────────────────────────────────
# 8 subjects total → 70% train / 15% val / 15% test  (by subject)
# train: subjects 1-5 (5/8 ≈ 63%), val: 6 (1/8 ≈ 13%), test: 7-8 (2/8 ≈ 25%)
# (adjust as needed for exact percentages)

TRAIN_SUBJECTS = [1, 2, 3, 4, 5]
VAL_SUBJECTS   = [6]
TEST_SUBJECTS  = [7, 8]

mask_train = np.isin(subjects, TRAIN_SUBJECTS)
mask_val   = np.isin(subjects, VAL_SUBJECTS)
mask_test  = np.isin(subjects, TEST_SUBJECTS)

X_train_raw, y_train = X[mask_train], y[mask_train]
X_val_raw,   y_val   = X[mask_val],   y[mask_val]
X_test_raw,  y_test  = X[mask_test],  y[mask_test]

print(f"Train samples : {len(y_train)}")
print(f"Val   samples : {len(y_val)}")
print(f"Test  samples : {len(y_test)}")

In [ ]:
# ── Normalise (fit only on train) ─────────────────────────────────────────────
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

print("Train mean (should be ~0):", X_train.mean().round(4))
print("Train std  (should be ~1):", X_train.std().round(4))

## 3. Train/Val/Test Split Distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

ACTIVITY_NAMES = [
    "Sitting", "Standing", "Lying Back", "Lying Right", "Ascending Stairs",
    "Descending Stairs", "Standing Elevator", "Moving Elevator",
    "Treadmill 4km/h", "Treadmill 4km/h Incline", "Treadmill 8km/h",
    "Cycling 70rpm", "Cycling 90rpm", "Cycling 110rpm",
    "Rowing 22", "Rowing 30", "Jumping", "Playing Basketball", "Stairs"
]

os.makedirs("results", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (split_y, title, color) in zip(
    axes,
    [(y_train, "Train", "#4C72B0"),
     (y_val,   "Validation", "#DD8452"),
     (y_test,  "Test", "#55A868")]
):
    counts = np.bincount(split_y, minlength=19)
    ax.barh(range(19), counts, color=color, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(19))
    ax.set_yticklabels(ACTIVITY_NAMES[:19], fontsize=8)
    ax.set_title(f"{title} ({len(split_y)} samples)", fontweight='bold')
    ax.set_xlabel("Count")
    ax.spines[['top','right']].set_visible(False)

plt.suptitle("Class Distribution per Split", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("results/split_distribution.png", bbox_inches='tight')
plt.show()
print("Saved: results/split_distribution.png")

## 4. Baseline Model — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='macro')

print(f"Test Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"F1-score (macro): {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=ACTIVITY_NAMES[:19]))

## 5. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=ACTIVITY_NAMES[:19],
    yticklabels=ACTIVITY_NAMES[:19],
    linewidths=0.4, linecolor='white',
    ax=ax
)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
ax.set_title("Confusion Matrix — Random Forest (normalised)", fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("results/confusion_matrix_rf.png", bbox_inches='tight')
plt.show()
print("Saved: results/confusion_matrix_rf.png")

## 6. Feature Importance (Top 20)

In [ ]:
# Feature names for 225 statistical features
STATS    = ["mean", "std", "min", "max", "range"]
CHANNELS = [f"ch{i+1:02d}" for i in range(45)]
feat_names = [f"{s}_{c}" for s in STATS for c in CHANNELS]  # 225 names

importances = rf.feature_importances_
top_idx = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    range(20),
    importances[top_idx][::-1],
    color=plt.cm.viridis(np.linspace(0.3, 0.9, 20))
)
ax.set_yticks(range(20))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=9)
ax.set_xlabel("Feature Importance (Gini)", fontsize=11)
ax.set_title("Top 20 Most Important Features — Random Forest", fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("results/feature_importance.png", bbox_inches='tight')
plt.show()
print("Saved: results/feature_importance.png")

## 7. Per-class Accuracy Bar Chart

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
sorted_idx = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d62728' if v < 0.70 else '#2ca02c' for v in per_class_acc[sorted_idx]]
ax.barh(
    range(19),
    per_class_acc[sorted_idx],
    color=colors, edgecolor='white'
)
ax.set_yticks(range(19))
ax.set_yticklabels([ACTIVITY_NAMES[i] for i in sorted_idx], fontsize=9)
ax.axvline(x=acc, color='navy', linestyle='--', label=f'Overall acc = {acc:.2f}')
ax.set_xlabel("Per-class Accuracy", fontsize=11)
ax.set_title("Per-class Accuracy — Random Forest", fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("results/per_class_accuracy.png", bbox_inches='tight')
plt.show()
print("Saved: results/per_class_accuracy.png")

## 8. Summary

In [ ]:
print("=" * 50)
print("WEEK 2 RESULTS SUMMARY")
print("=" * 50)
print(f"Total segments loaded : {len(y)}")
print(f"  Train               : {len(y_train)}")
print(f"  Validation          : {len(y_val)}")
print(f"  Test                : {len(y_test)}")
print()
print(f"Random Forest Baseline")
print(f"  Test Accuracy       : {acc*100:.2f}%")
print(f"  Macro F1-score      : {f1:.4f}")
print()
print("Plots saved to results/")
print("  - split_distribution.png")
print("  - confusion_matrix_rf.png")
print("  - feature_importance.png")
print("  - per_class_accuracy.png")
print()
print("Next week: 1D CNN and LSTM models")